# Medical Prescription OCR + MCP-Based Agentic RAG

Refactors the original hybrid RAG pipeline into proper **Model Context Protocol (MCP)** architecture.

## Architecture

```
Image → HandwritingOCR → GLiNER NER
                                ↓
                         MCP Server  ← tools registered via @mcp_server.list_tools / call_tool
                                ↓
                         MCP Client  ← agent communicates via JSON messages
                                ↓
                         ReAct Agent → Answer
```

| Component | Original | MCP version |
|---|---|---|
| Tool invocation | `self.tools.dispatch(name, args)` | `self.mcp_client.call_tool_sync(name, args)` |
| Tool discovery | Hard-coded `dispatch()` dict | `await client.list_tools()` at runtime |
| Tool schema | None | JSON Schema per tool |
| Protocol boundary | None | MCP messages |


## 1. Install Dependencies

In [ ]:
!pip install transformers==4.51.3 accelerate qwen-vl-utils -q
!pip install wikipedia-api requests sentence-transformers faiss-cpu -q
!pip install gliner rank-bm25 duckduckgo-search gradio -q
!pip install mcp -q   # MCP Python SDK

## 2. Imports & Constants

In [ ]:
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)
import asyncio, json, re, threading
import requests, numpy as np
from typing import Optional, List, Dict, Any

import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from gliner import GLiNER
from sentence_transformers import SentenceTransformer
import faiss, wikipediaapi
from rank_bm25 import BM25Okapi
try:
    from duckduckgo_search import DDGS
except ImportError:
    DDGS = None

# MCP SDK
from mcp.server import Server
from mcp import types as mcp_types

MODEL_NAME  = "Qwen/Qwen2-VL-2B-Instruct"
EMBED_MODEL = "all-MiniLM-L6-v2"
OPENFDA_BASE    = "https://api.fda.gov/drug/label.json"
RXNORM_BASE     = "https://rxnav.nlm.nih.gov/REST"
WIKI_USER_AGENT = "MedicalOCR-RAG/1.0 (research use)"
GLINER_LABELS   = ["drug","strength","dosage","frequency","duration","form","route"]
REACT_MAX_STEPS = 6

## 3. OCR Module (unchanged)

In [ ]:
class HandwritingOCR:
    def __init__(self, model_name=MODEL_NAME):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if self.device=="cuda" else torch.float32,
            device_map="auto")
        self.processor = AutoProcessor.from_pretrained(model_name)

    def extract_text(self, image_path: str, max_new_tokens: int = 512) -> str:
        messages = [{"role":"user","content":[
            {"type":"image","image":image_path},
            {"type":"text","text":(
                "Extract all handwritten text from this image. "
                "Correct obvious OCR spelling mistakes. "
                "Preserve line breaks and formatting. "
                "Return ONLY the final cleaned text.")}]}]
        text = self.processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
        image_inputs,video_inputs = process_vision_info(messages)
        inputs = self.processor(text=[text],images=image_inputs,videos=video_inputs,
                                padding=True,return_tensors="pt").to(self.device)
        generated_ids = self.model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False)
        trimmed = [out[len(inp):] for inp,out in zip(inputs.input_ids,generated_ids)]
        out = self.processor.batch_decode(trimmed,skip_special_tokens=True,
                                          clean_up_tokenization_spaces=True)[0]
        for prefix in ["Here is the extracted text:","Extracted text:","Output:","Answer:"]:
            if out.startswith(prefix): out = out[len(prefix):].strip()
        return out.strip()

## 4. MedicalKnowledgeRetriever (unchanged — runs inside MCP server)

In [ ]:
class MedicalKnowledgeRetriever:
    """
    Retrieval logic is unchanged from the original.
    In MCP architecture this class lives INSIDE the server.
    The agent never imports or calls it directly.
    """
    def __init__(self, embed_model: str = EMBED_MODEL):
        self.embedder  = SentenceTransformer(embed_model)
        self.wiki      = wikipediaapi.Wikipedia(language="en",user_agent=WIKI_USER_AGENT)
        self._passages: List[dict] = []
        self._index    = None
        self._bm25     = None

    def _fetch_wikipedia(self, term):
        page = self.wiki.page(term)
        if not page.exists(): page = self.wiki.page(f"{term} (medication)")
        if not page.exists(): return []
        words = page.summary.split()
        return [{"text":" ".join(words[i:i+80]),"source":f"Wikipedia:{page.title}","entity":term}
                for i in range(0,min(len(words),600),80) if " ".join(words[i:i+80])]

    def _fetch_openfda(self, drug_name):
        passages = []
        for stype in ["generic_name","brand_name"]:
            try:
                resp = requests.get(OPENFDA_BASE,
                    params={"search":f'openfda.{stype}:"{drug_name}"',"limit":1},timeout=8)
                result = resp.json().get("results",[])
                if result:
                    label = result[0]
                    for field in ["description","indications_and_usage","warnings",
                                  "dosage_and_administration","adverse_reactions",
                                  "drug_interactions","contraindications"]:
                        content = label.get(field,[])
                        if content:
                            passages.append({"text":" ".join(content)[:1200],
                                             "source":f"OpenFDA:{drug_name}",
                                             "entity":drug_name,"field":field})
                    break
            except Exception: pass
        return passages

    def _fetch_web_fallback(self, query):
        if DDGS is None: return []
        chunks = []
        try:
            with DDGS() as ddgs:
                for res in ddgs.text(query,max_results=4):
                    body = res.get("body","")
                    if len(body.split())>8:
                        chunks.append({"text":f"[{res.get('title','Web')}] {body}",
                                       "source":f"Web:{res.get('href','')}","entity":query})
        except Exception: pass
        return chunks

    def rxnorm_canonical(self, drug_name):
        try:
            resp = requests.get(f"{RXNORM_BASE}/drugs.json",params={"name":drug_name},timeout=6)
            for group in resp.json().get("drugGroup",{}).get("conceptGroup",[]):
                for concept in group.get("conceptProperties",[]):
                    if concept.get("name"): return concept["name"]
        except Exception: pass
        return None

    def _build_index(self, passages):
        if not passages: return
        self._passages = list(passages)
        texts = [p["text"] for p in passages]
        emb = self.embedder.encode(texts,normalize_embeddings=True,convert_to_numpy=True).astype("float32")
        self._index = faiss.IndexFlatIP(emb.shape[1])
        self._index.add(emb)
        self._bm25 = BM25Okapi([t.lower().split() for t in texts])

    def retrieve(self, query, top_k=4):
        if self._index is None or not self._passages: return []
        q_emb = self.embedder.encode([query],normalize_embeddings=True,convert_to_numpy=True).astype("float32")
        scores,indices = self._index.search(q_emb,top_k)
        dense = [{**self._passages[idx],"score":float(s)}
                 for s,idx in zip(scores[0],indices[0]) if 0<=idx<len(self._passages)]
        bm25s = self._bm25.get_scores(query.lower().split())
        sparse = [{**self._passages[i],"score":float(bm25s[i])}
                  for i in np.argsort(bm25s)[::-1][:top_k]]
        merged = {}
        for r in dense+sparse:
            k=r["text"]
            merged[k]={**r,"score":merged[k]["score"]+r["score"]} if k in merged else r
        return sorted(merged.values(),key=lambda x:x["score"],reverse=True)[:top_k]

    def load_for_entities(self, entities):
        existing = {p.get("entity") for p in self._passages}
        all_p = []
        for entity in entities:
            if entity in existing: continue
            wiki=self._fetch_wikipedia(entity); fda=self._fetch_openfda(entity)
            all_p.extend(wiki+fda)
            if not wiki and not fda:
                all_p.extend(self._fetch_web_fallback(f"{entity} drug information effects dosage"))
        if all_p: self._build_index(list(self._passages)+all_p)
        print(f"[RAG] Active passages: {len(self._passages)}")

## 5. MCP Server — Core Architectural Change

### What changed and why

**Original** used a plain Python `dispatch()` dict:
```python
tool_map = {
    "search_knowledge_base": lambda a: self.search_knowledge_base(...),
    ...
}
fn = tool_map.get(tool_name)
return fn(args)
```

**MCP version** registers tools with a schema on a protocol server:
```python
@mcp_server.list_tools()          # discovery endpoint — new
async def list_tools(): return TOOL_SCHEMAS

@mcp_server.call_tool()           # execution endpoint — replaces dispatch()
async def call_tool(name, arguments): ...
```

Key gains:
- Any MCP-compatible client can discover tools at runtime via `list_tools()`
- Each tool has a JSON Schema — input is validated, not just duck-typed
- The server can run in a separate process / container without code changes to the agent


In [ ]:
# ── Shared retriever singleton (lives inside the server) ──────────────
_retriever   = MedicalKnowledgeRetriever(EMBED_MODEL)
_wiki_client = wikipediaapi.Wikipedia(language="en",user_agent=WIKI_USER_AGENT)

mcp_server = Server("medical-rag-server")

# ── Tool schemas (NEW — original had none) ────────────────────────────
# Each tool declares its name, description, and JSON input schema.
# The agent can fetch these at runtime instead of having them hard-coded.
TOOL_SCHEMAS = [
    mcp_types.Tool(
        name="load_drug_knowledge",
        description="Fetch and index Wikipedia+OpenFDA data for a drug. ALWAYS call first.",
        inputSchema={"type":"object",
                     "properties":{"drug_name":{"type":"string"}},"required":["drug_name"]}),
    mcp_types.Tool(
        name="search_knowledge_base",
        description="Hybrid FAISS+BM25 search over indexed passages.",
        inputSchema={"type":"object",
                     "properties":{"query":{"type":"string"},"top_k":{"type":"integer","default":5}},
                     "required":["query"]}),
    mcp_types.Tool(
        name="fetch_openfda_field",
        description="Retrieve a specific OpenFDA label section (adverse_reactions, warnings, dosage_and_administration, contraindications, drug_interactions, indications_and_usage).",
        inputSchema={"type":"object",
                     "properties":{"drug_name":{"type":"string"},"field":{"type":"string"}},
                     "required":["drug_name","field"]}),
    mcp_types.Tool(
        name="fetch_wikipedia",
        description="Get the Wikipedia summary for a drug.",
        inputSchema={"type":"object","properties":{"drug_name":{"type":"string"}},"required":["drug_name"]}),
    mcp_types.Tool(
        name="lookup_rxnorm",
        description="Get canonical RxNorm name for a drug.",
        inputSchema={"type":"object","properties":{"drug_name":{"type":"string"}},"required":["drug_name"]}),
    mcp_types.Tool(
        name="web_search_engine",
        description="Fallback DuckDuckGo web search.",
        inputSchema={"type":"object","properties":{"query":{"type":"string"}},"required":["query"]}),
]


# ── list_tools endpoint (NEW) ─────────────────────────────────────────
# Original had no equivalent — tool names were hard-coded in dispatch().
# Now any client can discover available tools without reading source code.
@mcp_server.list_tools()
async def list_tools() -> list[mcp_types.Tool]:
    return TOOL_SCHEMAS


# ── call_tool endpoint (replaces MedicalTools.dispatch) ──────────────
# Original: fn = tool_map.get(tool_name); return fn(args)
# MCP:      structured JSON message → handler → structured JSON response
@mcp_server.call_tool()
async def call_tool(name: str, arguments: dict) -> list[mcp_types.TextContent]:
    def text(s): return [mcp_types.TextContent(type="text",text=str(s))]

    if name == "load_drug_knowledge":
        drug = arguments.get("drug_name","")
        _retriever.load_for_entities([drug])
        return text(f"Loaded real-time references for '{drug}'.")

    elif name == "search_knowledge_base":
        query = arguments.get("query","")
        top_k = int(arguments.get("top_k",5))
        passages = _retriever.retrieve(query,top_k=top_k)
        if not passages: return text("No matching records found in internal indexes.")
        return text("\n\n".join(
            f"[{i+1}] (Source:{p['source']})\n{p['text']}" for i,p in enumerate(passages)))

    elif name == "fetch_openfda_field":
        drug=arguments.get("drug_name","")
        field=arguments.get("field","adverse_reactions").strip().lower().replace(" ","_")
        valid={"adverse_reactions","warnings","contraindications","drug_interactions",
               "dosage_and_administration","indications_and_usage","description"}
        if field not in valid: return text(f"Invalid field '{field}'.")
        passages=_retriever._fetch_openfda(drug)
        if not passages: return text(f"No OpenFDA data found for '{drug}'.")
        matched=[p for p in passages if p.get("field")==field] or passages
        return text("\n\n".join(
            f"OpenFDA {p.get('field',field)} for {drug}:\n{p['text']}" for p in matched[:3]))

    elif name == "fetch_wikipedia":
        drug=arguments.get("drug_name","")
        for term in [drug, f"{drug} (medication)"]:
            page=_wiki_client.page(term)
            if page.exists(): return text(f"Wikipedia ({page.title}):\n{page.summary[:600]}")
        return text(f"No Wikipedia entry for '{drug}'.")

    elif name == "lookup_rxnorm":
        drug=arguments.get("drug_name","")
        canonical=_retriever.rxnorm_canonical(drug)
        return text(f"RxNorm: {canonical}" if canonical else "No RxNorm record.")

    elif name == "web_search_engine":
        query=arguments.get("query","")
        if DDGS is None: return text("duckduckgo-search not installed.")
        try:
            with DDGS() as ddgs:
                results=[r for r in ddgs.text(query,max_results=3)]
            if not results: return text("No web matches found.")
            return text("\n\n".join(f"URL:{r['href']}\nSnippet:{r['body']}" for r in results))
        except Exception as e: return text(f"Search error: {e}")

    else:
        return text(f"Unknown tool: {name}")

## 6. MCP Client — Agent's Interface to the Server

### What changed and why

**Original** — agent owned `MedicalTools` and called Python methods:
```python
class ReActAgent:
    def __init__(self, qwen, tools: MedicalTools):
        self.tools = tools

    # Inside run():
    observation = self.tools.dispatch(action, args)  # Python method call
```

**MCP version** — agent holds an `MCPClient` and sends protocol messages:
```python
class ReActAgent:
    def __init__(self, qwen, mcp_client: MCPClient):
        self.mcp_client = mcp_client

    # Inside run():
    observation = self.mcp_client.call_tool_sync(action, args)  # MCP message
```

The client also exposes `list_tools()` — the agent can enumerate available
tools at runtime without knowing them in advance.

> In production, replace the in-process implementation with:
> ```python
> async with mcp_stdio.stdio_client(server_params) as (read, write):
>     async with ClientSession(read, write) as session:
>         await session.initialize()
>         result = await session.call_tool(name, arguments)
> ```
> The agent code is identical either way.


In [ ]:
class MCPClient:
    """
    In-process MCP client (no subprocess needed in notebooks).
    Routes tool calls through the registered MCP server handlers above.
    Interface is identical to a real remote MCP client.
    """

    async def list_tools(self) -> list[dict]:
        """
        NEW: tool discovery via MCP.
        Original had no equivalent — names were hard-coded in dispatch().
        """
        tools = await list_tools()
        return [{"name":t.name,"description":t.description,"inputSchema":t.inputSchema}
                for t in tools]

    async def call_tool(self, name: str, arguments: dict) -> str:
        """
        Send a tool-call message through the MCP protocol.
        Replaces: self.tools.dispatch(name, args)
        """
        results = await call_tool(name, arguments)
        return "\n".join(r.text for r in results if hasattr(r,"text"))

    def call_tool_sync(self, name: str, arguments: dict) -> str:
        """Synchronous wrapper for use inside the ReAct loop (Colab/Jupyter safe)."""
        try:
            loop = asyncio.get_event_loop()
            if loop.is_running():
                holder = []
                def _run():
                    nl = asyncio.new_event_loop()
                    holder.append(nl.run_until_complete(self.call_tool(name,arguments)))
                    nl.close()
                t = threading.Thread(target=_run); t.start(); t.join()
                return holder[0] if holder else "Error: no result"
            else:
                return loop.run_until_complete(self.call_tool(name,arguments))
        except Exception as e:
            return f"MCP call error: {e}" 

## 7. ReAct Agent — Updated to Use MCP Client

### What changed

The ReAct loop logic (Thought → Action → Args → Observation, prompt building,
parsing, heuristics) is **completely unchanged**.

The **only structural change** is one line in `_preload_drugs` and in `run()`:

```python
# BEFORE (original)
observation = self.tools.dispatch(action, args)

# AFTER (MCP)
observation = self.mcp_client.call_tool_sync(action, args)
```

And the constructor:
```python
# BEFORE
def __init__(self, qwen, tools: MedicalTools):

# AFTER
def __init__(self, qwen, mcp_client: MCPClient):
```


In [ ]:
REACT_SYSTEM = """You are a precise medical reasoning assistant with access to tools.
You operate in a strict Thought → Action → Args loop.
Stop generating immediately after writing Args. Do NOT write the Observation yourself.

Available tools:
1. load_drug_knowledge  - ALWAYS call this first for any drug.
   Args: {"drug_name": "name"}
2. search_knowledge_base - Search indexed passages.
   Args: {"query": "text", "top_k": "5"}
3. fetch_openfda_field  - Get a specific OpenFDA label section.
   Choices: adverse_reactions, warnings, contraindications,
            drug_interactions, dosage_and_administration, indications_and_usage
   Args: {"drug_name": "name", "field": "field_name"}
4. fetch_wikipedia      - Get Wikipedia summary.
   Args: {"drug_name": "name"}
5. lookup_rxnorm        - Get canonical drug name.
   Args: {"drug_name": "name"}
6. finish               - Return final answer.
   Args: {"answer": "Your complete synthesis here"}

Strict output format:
Thought: <reasoning>
Action: <tool_name>
Args: {"key": "value"}
"""

REACT_FEW_SHOT = """
Example:
Question: What are the side effects of Amoxicillin?
Thought: Load knowledge for Amoxicillin first.
Action: load_drug_knowledge
Args: {"drug_name": "Amoxicillin"}
Observation: Loaded real-time references for 'Amoxicillin'.
Thought: Fetch adverse reactions from OpenFDA.
Action: fetch_openfda_field
Args: {"drug_name": "Amoxicillin", "field": "adverse_reactions"}
Observation: OpenFDA adverse_reactions: Nausea, vomiting, diarrhea, rash...
Thought: Sufficient information collected.
Action: finish
Args: {"answer": "Common side effects of Amoxicillin include nausea, vomiting, diarrhea, and rash."}
"""


class QwenInference:
    def __init__(self, model, processor):
        self.model = model; self.processor = processor
        self.device = next(model.parameters()).device

    def generate(self, prompt, max_new_tokens=512):
        messages=[{"role":"user","content":[{"type":"text","text":prompt}]}]
        text=self.processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
        inputs=self.processor(text=[text],return_tensors="pt").to(self.device)
        with torch.no_grad():
            ids=self.model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False)
        trimmed=ids[0][inputs.input_ids.shape[1]:]
        return self.processor.decode(trimmed,skip_special_tokens=True).strip()


class ReActAgent:
    """
    ReAct agent — logic unchanged, tool calls routed through MCPClient.
    Original: self.tools.dispatch(action, args)   ← direct Python
    MCP:      self.mcp_client.call_tool_sync(action, args)  ← protocol
    """

    def __init__(self, qwen: QwenInference, mcp_client: MCPClient):  # ← was MedicalTools
        self.qwen = qwen
        self.mcp_client = mcp_client

    @staticmethod
    def _parse_action(text):
        am=re.search(r"Action:\s*([A-Za-z0-9_]+)",text)
        argm=re.search(r"Args:\s*(\{.*?\})",text,re.DOTALL)
        if not am: return None,{}
        args={}
        if argm:
            try: args=json.loads(argm.group(1).strip())
            except Exception:
                try: args=eval(argm.group(1).strip())
                except Exception: pass
        return am.group(1).strip(), args

    @staticmethod
    def _build_prompt(question, drug_names, trajectory):
        prompt=f"{REACT_SYSTEM}\n{REACT_FEW_SHOT}\n"
        if drug_names: prompt+=f"Context: Prescription contains: {', '.join(drug_names)}\n\n"
        prompt+=f"Current Task:\nQuestion: {question}\n\n"
        for step in trajectory:
            prompt+=f"Thought: {step['thought']}\nAction: {step['action']}\nArgs: {json.dumps(step['args'])}\n"
            if step["action"]!="finish": prompt+=f"Observation: {step['observation']}\n\n"
        return prompt+"Thought:"

    def _preload_drugs(self, drug_names, trajectory):
        already={s["args"].get("drug_name") for s in trajectory if s["action"]=="load_drug_knowledge"}
        for drug in drug_names:
            if drug and drug not in already:
                print(f"[ReAct/MCP] Pre-loading '{drug}'…")
                # ── MCP CALL ──────────────────────────────────────────────────────
                obs = self.mcp_client.call_tool_sync("load_drug_knowledge",{"drug_name":drug})
                trajectory.append({"thought":f"Pre-loading references for {drug} via MCP.",
                                    "action":"load_drug_knowledge",
                                    "args":{"drug_name":drug},"observation":obs})

    @staticmethod
    def _heuristic_first_action(question, drug_names):
        q=question.lower()
        if not drug_names: return None,{}
        drug=drug_names[0]
        field_map={
            ("side effect","adverse","reaction","risk"):"adverse_reactions",
            ("warning","caution","precaution"):"warnings",
            ("contraindic","avoid"):"contraindications",
            ("interact","combination","together"):"drug_interactions",
            ("dose","dosage","how much","how many","administration"):"dosage_and_administration",
            ("use","treat","indicat","purpose","for what"):"indications_and_usage",
        }
        for kws,field in field_map.items():
            if any(kw in q for kw in kws):
                return "fetch_openfda_field",{"drug_name":drug,"field":field}
        return "search_knowledge_base",{"query":question,"top_k":"5"}

    def run(self, question, drug_names=None):
        drug_names=drug_names or []; trajectory=[]
        self._preload_drugs(drug_names,trajectory)
        final_answer=None

        for step_num in range(1,REACT_MAX_STEPS+1):
            print(f"[ReAct/MCP] Step {step_num}/{REACT_MAX_STEPS}")
            raw=self.qwen.generate(self._build_prompt(question,drug_names,trajectory),max_new_tokens=300)
            if "Observation:" in raw: raw=raw.split("Observation:")[0].strip()

            thought="Reasoning about the question."
            if "Action:" in raw: thought=raw.split("Action:")[0].replace("Thought:","").strip()
            elif "Thought:" in raw: thought=raw.replace("Thought:","").strip()

            action,args=self._parse_action(raw)
            if action is None:
                action,args=self._heuristic_first_action(question,drug_names)
                if action is None:
                    obs_texts=" ".join(s["observation"][:300] for s in trajectory if s.get("observation"))
                    final_answer=obs_texts or "Could not find specific information."
                    break
                thought=f"Heuristic: applying {action}."

            if action.lower()=="finish":
                final_answer=args.get("answer",thought)
                trajectory.append({"thought":thought,"action":"finish","args":args,"observation":""})
                break

            # ── MCP CALL (was: self.tools.dispatch(action, args)) ─────────────
            print(f"[MCP Client] → {action}({args})")
            observation=self.mcp_client.call_tool_sync(action,args)
            print(f"[MCP Server] ← {observation[:200]}…")
            trajectory.append({"thought":thought,"action":action,"args":args,"observation":observation})

        if final_answer is None:
            obs_texts=" ".join(s.get("observation","")[:300] for s in trajectory)
            final_answer=obs_texts or "Max steps reached."
        return {"answer":final_answer,"trajectory":trajectory,"sources":[]}

## 8. NERCorrector (unchanged)

In [ ]:
class NERCorrector:
    def __init__(self, retriever: MedicalKnowledgeRetriever):
        self.retriever=retriever

    def _rxnorm_match(self, drug_name):
        canonical=self.retriever.rxnorm_canonical(drug_name)
        wiki=wikipediaapi.Wikipedia(language="en",user_agent=WIKI_USER_AGENT)
        page=wiki.page(drug_name)
        return {"original":drug_name,"corrected":canonical or drug_name,
                "sources":(["RxNorm"] if canonical else [])+(["Wikipedia"] if page.exists() else [])}

    def correct_entities(self, entities):
        corrected=[]
        for ent in entities:
            correction=(self._rxnorm_match(ent["text"]) if ent.get("label")=="drug"
                        else {"original":ent["text"],"corrected":ent["text"],"sources":[]})
            corrected.append({**ent,"correction":correction})
        return corrected

## 9. MedicalPipeline — Wires MCP Client to Agent

### What changed

```python
# BEFORE
self.tools = MedicalTools(self.retriever)
self.agent = ReActAgent(self.qwen, self.tools)

# AFTER
self.mcp_client = MCPClient()
self.agent      = ReActAgent(self.qwen, self.mcp_client)
```

Knowledge loading also goes through MCP now:
```python
# BEFORE
self.retriever.load_for_entities(drug_terms)

# AFTER
self.mcp_client.call_tool_sync("load_drug_knowledge", {"drug_name": drug})
```


In [ ]:
class MedicalPipeline:
    def __init__(self, ocr_model=MODEL_NAME, embed_model=EMBED_MODEL):
        print("[Pipeline] Loading OCR / reasoning model (Qwen2-VL)...")
        self.ocr=HandwritingOCR(ocr_model)
        self.qwen=QwenInference(self.ocr.model,self.ocr.processor)

        print("[Pipeline] Loading GLiNER NER model...")
        self.ner_model=GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

        print("[Pipeline] Initialising MCP client + server...")
        self.corrector=NERCorrector(_retriever)

        # ── KEY CHANGE: MCPClient replaces MedicalTools ───────────────
        self.mcp_client=MCPClient()
        self.agent=ReActAgent(self.qwen,self.mcp_client)

    def run_ocr(self,image_path):
        print(f"[OCR] Processing {image_path}...")
        text=self.ocr.extract_text(image_path)
        print(f"[OCR] Result:\n{text}\n"); return text

    def run_ner(self,text):
        entities=self.ner_model.predict_entities(text,labels=GLINER_LABELS,threshold=0.4)
        formatted=[{"text":e["text"],"label":e["label"].lower(),"start":e["start"],
                    "end":e["end"],"score":round(e["score"],4)} for e in entities]
        print(f"[NER] Found {len(formatted)} entities:")
        for e in formatted: print(f"  - {e['text']} ({e['label']}) [{e['score']}]")
        return formatted

    def load_knowledge(self,entities):
        """
        Knowledge loading now goes through MCP rather than calling retriever directly.
        Original: self.retriever.load_for_entities(drug_terms)
        MCP:      self.mcp_client.call_tool_sync('load_drug_knowledge', {'drug_name': drug})
        """
        drug_terms=list({e["text"] for e in entities if e["label"]=="drug"})
        if not drug_terms: print("[RAG] No DRUG entities — skipping."); return
        for drug in drug_terms:
            print(f"[Pipeline/MCP] Loading '{drug}'...")
            self.mcp_client.call_tool_sync("load_drug_knowledge",{"drug_name":drug})

    def run_correction(self,entities):
        corrected=self.corrector.correct_entities(entities)
        for e in corrected:
            c=e["correction"]
            status=f"→ '{c['corrected']}'" if c["corrected"]!=c["original"] else "— unchanged"
            print(f"[Correction] '{c['original']}' {status}")
        return corrected

    def to_structured_json(self,corrected_entities):
        result={label:[] for label in GLINER_LABELS}
        for ent in corrected_entities:
            label=ent["label"]
            if label in result:
                display=ent["correction"]["corrected"] if label=="drug" else ent["text"]
                result[label].append({"value":display,"original_ocr":ent["text"],
                                      "sources":ent["correction"].get("sources",[])})
        return result

    def process_image(self,image_path):
        text=self.run_ocr(image_path); entities=self.run_ner(text)
        self.load_knowledge(entities); corrected=self.run_correction(entities)
        return {"raw_text":text,"entities_raw":entities,
                "entities_corrected":corrected,"structured_prescription":self.to_structured_json(corrected)}

    def process_text(self,text):
        entities=self.run_ner(text); self.load_knowledge(entities)
        corrected=self.run_correction(entities)
        return {"raw_text":text,"entities_raw":entities,
                "entities_corrected":corrected,"structured_prescription":self.to_structured_json(corrected)}

    def ask(self,question,structured_prescription=None):
        drug_names=[]
        if structured_prescription:
            for entry in structured_prescription.get("drug",[]):
                name=entry.get("value") or entry.get("original_ocr","")
                if name: drug_names.append(name)
        print(f"[Agent] Question: {question}\n[Agent] Known drugs: {drug_names}")
        return self.agent.run(question,drug_names=drug_names)

## 10. Gradio UI (unchanged)

In [ ]:
import gradio as gr
import pandas as pd
import tempfile, os

pipeline = MedicalPipeline()

def run_pipeline(image, question):
    if image is None:
        return None,"No image uploaded.",pd.DataFrame(),"{}","","",""
    temp_path=None
    try:
        with tempfile.NamedTemporaryFile(suffix=".png",delete=False) as tmp:
            temp_path=tmp.name; image.save(tmp,format="PNG")
        result=pipeline.process_image(temp_path)
        raw_text=result.get("raw_text","")
        ner_rows=[{"Label":e.get("label",""),"Original":e.get("correction",{}).get("original",""),
                   "Corrected":e.get("correction",{}).get("corrected",""),"Confidence":e.get("score","")}
                  for e in result.get("entities_corrected",[])]
        ner_df=pd.DataFrame(ner_rows)
        structured_json=json.dumps(result.get("structured_prescription",{}),indent=2)
        answer_text=source_text=agent_trace=""
        if question.strip():
            qa=pipeline.ask(question,result.get("structured_prescription"))
            answer_text=qa.get("answer","")
            source_text="\n".join(qa.get("sources",[])) or "N/A"
            lines=[]
            for i,step in enumerate(qa.get("trajectory",[]),1):
                lines.append(f"── Step {i} ──")
                if step.get("thought"): lines.append(f"Thought: {step['thought']}")
                lines.append(f"Action:  {step['action']}({json.dumps(step.get('args',{}))})")
                if step.get("observation"):
                    obs=step["observation"][:300]
                    lines.append(f"Obs:     {obs}{'...' if len(step['observation'])>300 else ''}")
            agent_trace="\n".join(lines)
        return image,raw_text,ner_df,structured_json,answer_text,source_text,agent_trace
    except Exception as e:
        import traceback
        return image,f"Error: {e}\n{traceback.format_exc()}",pd.DataFrame(),"{}","","",""
    finally:
        if temp_path and os.path.exists(temp_path):
            try: os.unlink(temp_path)
            except Exception: pass

with gr.Blocks(title="Medical OCR + MCP Agentic RAG") as demo:
    gr.Markdown("# Medical Prescription Pipeline — MCP Agentic RAG\nAgent calls tools via Model Context Protocol.")
    with gr.Row():
        with gr.Column(scale=1):
            image_input=gr.Image(type="pil",label="Upload Prescription Image")
            question_input=gr.Textbox(label="Ask a Medical Question",
                placeholder="e.g. What are the side effects of the medicines prescribed?")
            run_btn=gr.Button("Run Pipeline",variant="primary")
        with gr.Column(scale=1):
            image_preview=gr.Image(label="Uploaded Image")
            ocr_output=gr.Textbox(label="Step 1: OCR Output",lines=8)
    ner_output=gr.Dataframe(headers=["Label","Original","Corrected","Confidence"],
                            label="Step 2: NER + Corrections",interactive=False)
    structured_output=gr.Code(label="Step 3: Structured Prescription JSON",language="json")
    with gr.Row():
        answer_output=gr.Textbox(label="Step 4: Agent Answer (via MCP)",lines=8)
        source_output=gr.Textbox(label="Sources Used",lines=8)
    agent_trace_output=gr.Textbox(label="Agent Reasoning Trace (MCP calls)",lines=20)
    run_btn.click(fn=run_pipeline,inputs=[image_input,question_input],
                  outputs=[image_preview,ocr_output,ner_output,structured_output,
                           answer_output,source_output,agent_trace_output])
demo.launch(debug=True,share=True)